# 08. Pattern Matching: KMP, Z-Function, Rabin-Karp

Find a **pattern** inside a **text**. The naive way is O(n·m). These algorithms make it **O(n + m)**.

**Learning order:** NAIVE → KMP IDEA → LPS TABLE → KMP SEARCH → Z-FUNCTION → RABIN-KARP → APPLICATIONS → INTERVIEW THINKING

## 1. Naive search

Try every start position and compare character by character. Worst case `text = "aaaa...a"`, `pattern = "aaa...b"`: O(n·m). After a mismatch it **forgets** everything it learned.

In [1]:
def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    for i in range(n - m + 1):
        if text[i:i + m] == pattern:
            return i
    return -1

print(naive_search("hello world", "world"), naive_search("hello", "xyz"))   # 6 -1

6 -1


## 2. KMP: never move backwards in the text

**Idea:** when a mismatch happens after matching `k` characters, we already know the text equals `pattern[:k]`. Instead of restarting, **jump the pattern to the longest proper prefix of `pattern[:k]` that is also its suffix**. That value is stored in the **LPS array** ("longest prefix which is also suffix").

`lps[i]` = length of the longest proper prefix of `pattern[:i+1]` that is also a suffix of it.

Example `"aabaaab"` → `[0, 1, 0, 1, 2, 2, 3]`.

In [2]:
def build_lps(pattern):
    lps = [0] * len(pattern)
    k = 0                                                   # length of the current matched prefix
    for i in range(1, len(pattern)):
        while k and pattern[i] != pattern[k]:
            k = lps[k - 1]                                  # fall back, never recompare
        if pattern[i] == pattern[k]:
            k += 1
        lps[i] = k
    return lps

print(build_lps("aabaaab"), build_lps("abab"), build_lps("abcdabca"))   # [0, 1, 0, 1, 2, 2, 3] [0, 0, 1, 2] [0, 0, 0, 0, 1, 2, 3, 1]

[0, 1, 0, 1, 2, 2, 3] [0, 0, 1, 2] [0, 0, 0, 0, 1, 2, 3, 1]


## 3. KMP search

Walk through the text with a pointer `k` into the pattern. On a mismatch, `k = lps[k - 1]`. When `k == len(pattern)` we found a match ending at `i`. The text index **never goes back**, and `k` falls back at most as much as it advanced: **O(n + m)**.

In [3]:
def kmp_search(text, pattern):                              # returns all start indexes
    if not pattern:
        return []
    lps, k, res = build_lps(pattern), 0, []
    for i, ch in enumerate(text):
        while k and ch != pattern[k]:
            k = lps[k - 1]
        if ch == pattern[k]:
            k += 1
        if k == len(pattern):
            res.append(i - k + 1)
            k = lps[k - 1]
    return res

print(kmp_search("ABABDABACDABABCABAB", "ABABCABAB"))   # [10]
print(kmp_search("aaaaa", "aa"))                        # [0, 1, 2, 3]

[10]
[0, 1, 2, 3]


## 4. Z-function

`z[i]` = length of the longest substring starting at `i` that matches a **prefix of the string**. Keep a window `[l, r)` of the rightmost match; inside it reuse `z[i - l]`. O(n).

**Pattern search with Z:** build `pattern + "#" + text`; every position with `z == len(pattern)` is a match.

In [4]:
def z_function(s):
    n = len(s)
    z = [0] * n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def z_search(text, pattern):
    combined = pattern + "#" + text
    z = z_function(combined)
    return [i - len(pattern) - 1 for i in range(len(combined)) if z[i] == len(pattern)]

print(z_function("aabxaab"))                    # [0, 1, 0, 0, 3, 1, 0]
print(z_search("ABABDABACDABABCABAB", "ABABCABAB"))   # [10]

[0, 1, 0, 0, 3, 1, 0]
[10]


## 5. Rabin-Karp (rolling hash)

Turn each window of the text into a number (**hash**) and compare numbers. The next window's hash is computed from the previous one in O(1): remove the leftmost character, shift, add the new one. On a hash match, **verify the characters** (different strings can share a hash: a collision). Average O(n + m), worst O(n·m) with many collisions.

Good for **multiple patterns** (store pattern hashes in a set) and for substring problems like *Longest Duplicate Substring* with binary search.

In [5]:
def rabin_karp(text, pattern, base=256, mod=1_000_000_007):
    n, m = len(text), len(pattern)
    if m > n:
        return []
    high = pow(base, m - 1, mod)                            # weight of the leftmost character
    hp = ht = 0
    for i in range(m):
        hp = (hp * base + ord(pattern[i])) % mod
        ht = (ht * base + ord(text[i])) % mod
    res = []
    for i in range(n - m + 1):
        if hp == ht and text[i:i + m] == pattern:           # verify to rule out collisions
            res.append(i)
        if i + m < n:
            ht = ((ht - ord(text[i]) * high) * base + ord(text[i + m])) % mod
    return res

print(rabin_karp("ABABDABACDABABCABAB", "ABABCABAB"), rabin_karp("aaaaa", "aa"))   # [10] [0, 1, 2, 3]

[10] [0, 1, 2, 3]


## 6. Applications of the LPS table

- **Repeated Substring Pattern:** `s` is built by repeating a block iff `lps[-1] > 0` and `n % (n - lps[-1]) == 0`.
- **Shortest Palindrome** (add characters only in front): the longest palindromic **prefix** of `s` = the LPS of `s + "#" + reverse(s)`. Add the reverse of the remaining suffix in front.
- **Smallest period** of a string = `n - lps[-1]`.

In [6]:
def repeated_substring_pattern(s):
    n = len(s)
    lps = build_lps(s)
    return lps[-1] > 0 and n % (n - lps[-1]) == 0

def shortest_palindrome(s):
    lps = build_lps(s + "#" + s[::-1])
    return s[::-1][:len(s) - lps[-1]] + s

print(repeated_substring_pattern("abab"), repeated_substring_pattern("aba"), repeated_substring_pattern("abcabcabcabc"))   # True False True
print(shortest_palindrome("aacecaaa"), shortest_palindrome("abcd"))   # aaacecaaa dcbabcd

True False True
aaacecaaa dcbabcd


## 7. Which one to use?

| | Time | Extra space | Notes |
|---|---|---|---|
| Naive | O(n·m) | O(1) | fine for tiny inputs |
| **KMP** | O(n + m) | O(m) | guaranteed, most asked |
| Z-function | O(n + m) | O(n + m) | simple to code once you know it |
| Rabin-Karp | O(n + m) avg | O(1) | multiple patterns, 2D patterns, rolling hash tricks |
| Python `text.find(p)` | fast in C (two-way + heuristics) | | use in real code |

## 8. Interview thinking

**Say:** "Naive search can degrade to O(n·m). KMP precomputes, for every prefix of the pattern, the longest border (LPS), so on a mismatch I fall back inside the pattern instead of re-reading the text. O(n + m)."

**Common mistakes:** wrong LPS indexing (`lps[k - 1]`), forgetting `k = lps[k - 1]` after a full match (to find overlapping matches), no collision verification in Rabin-Karp, negative modulo values in the rolling update (in Python `%` is fine).

## Quick Revision

- KMP: LPS = longest proper prefix that is also a suffix. Mismatch → `k = lps[k-1]`. O(n + m).
- Z: matches with the prefix, search on `pattern + "#" + text`.
- Rabin-Karp: rolling hash, verify on match.
- LPS applications: repeated substring, shortest palindrome, smallest period.